In [1]:
import pandas as pd
import numpy as np

# 1. LOAD THE DATASET
df = pd.read_csv('dirty_cafe_sales.csv')

print(f"Initial dimensions (Raw): {df.shape[0]} rows x {df.shape[1]} columns\n")
display(df.head())

Initial dimensions (Raw): 10000 rows x 8 columns



,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [2]:
# 2. DIAGNOSTIC AND TREATMENT OF NULLS / INVALID VALUES
# This dataset often contains text strings like 'ERROR', 'UNKNOWN', or empty spaces
invalid_mask = ['ERROR', 'UNKNOWN', 'UNKNOWN ', '', ' ', 'Error', 'Unknown']
df = df.replace(invalid_mask, np.nan)

print("--- Real nulls detected after cleaning error strings ---")
print(df.isnull().sum())

--- Real nulls detected after cleaning error strings ---
Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64


In [3]:
# 3. REMOVING DUPLICATES
# Drop completely duplicate rows or repeated transaction records
rows_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
rows_after = len(df)

print(f"Removed {rows_before - rows_after} duplicate rows.")

Removed 0 duplicate rows.


In [4]:
# 4. DATA TYPE CONVERSION

numeric_cols = ['Quantity', 'Price Per Unit', 'Total Spent']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Standardize dates if the column exists
if 'Transaction Date' in df.columns:
    df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

In [5]:
# 5. MATHEMATICAL VALIDATION AND RECALCULATION (Business Rule)
# Rule: Total Spent = Quantity * Price Per Unit
# If Total Spent is missing but we have Quantity and Price, recalculate it.
mask_recalc = df['Total Spent'].isnull() & df['Quantity'].notnull() & df['Price Per Unit'].notnull()
df.loc[mask_recalc, 'Total Spent'] = df.loc[mask_recalc, 'Quantity'] * df.loc[mask_recalc, 'Price Per Unit']

# If Price Per Unit is missing, infer it by dividing Total Spent by Quantity
mask_price = df['Price Per Unit'].isnull() & df['Total Spent'].notnull() & df['Quantity'].notnull()
df.loc[mask_price, 'Price Per Unit'] = df.loc[mask_price, 'Total Spent'] / df.loc[mask_price, 'Quantity']

print("Mathematical recalculation completed successfully.")

Mathematical recalculation completed successfully.


In [6]:
# 6. CATEGORICAL TEXT STANDARDIZATION
# Convert text to title case (e.g., "coffee" -> "Coffee") and strip extra spaces
text_cols = df.select_dtypes(include=['object', 'str']).columns
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()
    df[col] = df[col].replace({'Nan': np.nan, 'None': np.nan})

# 7. EXPORT THE CLEAN DATASET
df.to_csv('cafe_sales_clean.csv', index=False)
print("\n--- Process Finished! ---")
print(f"Final dimensions: {df.shape[0]} rows x {df.shape[1]} columns")
print("File 'cafe_sales_clean.csv' saved successfully, ready for Power BI or GitHub.")
display(df.head())


--- Process Finished! ---
Final dimensions: 10000 rows x 8 columns
File 'cafe_sales_clean.csv' saved successfully, ready for Power BI or GitHub.


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,Txn_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,Txn_4977031,Cake,4.0,3.0,12.0,Cash,In-Store,2023-05-16
2,Txn_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-Store,2023-07-19
3,Txn_7034554,Salad,2.0,5.0,10.0,NaN,NaN,2023-04-27
4,Txn_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-Store,2023-06-11
